# Step 5: Find domains

This step computes domain analysis and compares T cell phenotypes across domains. Adapted from https://scimap.xyz/tutorials/md/spatial_lda_scimap/

## Setup and imports

In [ ]:
# Imports
import sys
print(sys.executable)
from copy import deepcopy
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import os
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
from scipy.stats import mannwhitneyu, kruskal, friedmanchisquare, wilcoxon, zscore, norm
from statsmodels.stats.multitest import fdrcorrection, multipletests
from sklearn.neighbors import NearestNeighbors
import matplotlib as mpl
from scipy import sparse
import scimap as sm
import anndata as ad
from pathlib import Path
import random as random

In [ ]:
# Reproducibility settings
sc.settings.verbosity = 3
np.random.seed(26)

In [ ]:
# Path config
indir = '/Users/yyj/Doc/1_dod_dec25/processed_data/' #indir = '/path/to/xenium/raw_data'
outdir = '/Users/yyj/Doc/1_dod_dec25/outdir/' #outdir = '/path/to/integrated/processed_data'
figdir = '/Users/yyj/Doc/1_dod_dec25/figures/'

In [ ]:
# Figure settings
plt.rcParams['savefig.transparent'] = True
plt.rcParams['savefig.dpi'] = 600
mpl.rcParams["font.family"] = "Helvetica"
mpl.rcParams['font.size'] = 6
sc.settings.figdir = figdir

## Load data

In [ ]:
adata = sc.read_h5ad(indir + 'xenium_integrated_labeled.h5ad')

## Compute domains

In [ ]:
adata.obs['X_centroid'] = adata.obsm['spatial'][:, 0]
adata.obs['Y_centroid'] = adata.obsm['spatial'][:, 1]

In [ ]:
adata = sm.tl.spatial_count(
    adata, 
    phenotype='celltypes_all',  # Verify this column name!
    method='radius', 
    radius=80, 
    imageid='sample',
    x_coordinate='X_centroid',
    y_coordinate='Y_centroid',
    label='spatial_count'
)

In [ ]:
adata = sm.tl.spatial_cluster(adata, df_name='spatial_count', method='kmeans', k=20, label='neigh_kmeans')

In [ ]:
# Force natural numeric order for neigh_kmeans
nk = pd.to_numeric(adata.obs["neigh_kmeans"], errors="coerce")
order = sorted(nk.dropna().unique().astype(int))

adata.obs["neigh_kmeans"] = pd.Categorical(
    nk.astype("Int64").astype(str),
    categories=[str(i) for i in order],
    ordered=True
)

In [ ]:
# Build celltype palette from adata.uns
if "celltype_colors" in adata.uns and pd.api.types.is_categorical_dtype(adata.obs["celltype"]):
    ct_cats = adata.obs["celltype"].cat.categories
    ct_cols = adata.uns["celltype_colors"]
    celltype_palette = dict(zip(ct_cats, ct_cols))
else:
    celltype_palette = {c: "#E6E6E6" for c in pd.unique(adata.obs["celltype"])}

In [ ]:
# Build proportions table
df = adata.obs[["neigh_kmeans", "celltype"]].copy()
df["neigh_kmeans"] = df["neigh_kmeans"].astype(str)  # normalize keys for robust ordering

ct = (
    df.groupby(["neigh_kmeans", "celltype"])
      .size()
      .unstack(fill_value=0)
)

ct = ct.div(ct.sum(axis=1), axis=0).fillna(0)

# Desired celltype stack order
desired_order = ["Neuroblast", "Endothelial", "Fibroblast", "Schwann", "Macrophage", "B", "T"]
present = [c for c in desired_order if c in ct.columns]
extras = [c for c in ct.columns if c not in present]
ct = ct[present + extras]

In [ ]:
# Custom bar order:
# head = sorted by Neuroblast proportion (desc)
# tail = fixed 8,9,0,4 if present
fixed_tail = ["8", "9", "0", "4"]

if "Neuroblast" in ct.columns:
    neuro_prop = ct["Neuroblast"]
else:
    neuro_prop = pd.Series(0.0, index=ct.index)

head_candidates = [k for k in ct.index if k not in fixed_tail]
head_sorted = neuro_prop.loc[head_candidates].sort_values(ascending=False).index.tolist()

tail_present = [k for k in fixed_tail if k in ct.index]
tail_missing = [k for k in fixed_tail if k not in ct.index]
if tail_missing:
    print(f"Warning: missing neigh_kmeans in data (cannot place at tail): {tail_missing}")

final_order = head_sorted + tail_present
ct = ct.reindex(final_order)

In [ ]:
# Plot domains
fig, ax = plt.subplots(figsize=(9/2.54, 8.5/2.54))

ct.plot.bar(
    stacked=True,
    color=[celltype_palette.get(ct_name, "#E6E6E6") for ct_name in ct.columns],
    edgecolor="none",
    linewidth=0,
    width=0.95,
    ax=ax,
    legend=False
)

x = np.arange(len(ct.index))
ax.set_xticks(x)
ax.set_xticklabels(ct.index, rotation=0, ha="right", fontsize=5)
ax.set_xlim(-0.5, len(ct.index) - 0.5)

ax.set_ylim(0, 1.0)
yt = np.linspace(0, 1, 6)
ax.set_yticks(yt)
ax.set_yticklabels([f"{v:.1f}" for v in yt], rotation=0, fontsize=5)

ax.set_xlabel("Domain", fontsize=5)
ax.set_ylabel("Proportion of Cells", fontsize=5)

# keep visible tick marks
ax.tick_params(axis="both", which="both", left=True, bottom=True, width=0.5, length=2)

# Add proportion labels on each segment
for i, idx in enumerate(ct.index):
    bottom = 0.0
    for col in ct.columns:
        val = ct.loc[idx, col]
        if val < 0.03:
            bottom += val
            continue
        ax.text(
            i, bottom + val / 2,
            f"{val:.2f}",
            ha="center", va="center",
            fontsize=5, color="black"
        )
        bottom += val

# spines style (match your clean style but keep bottom visible)
ax.spines["bottom"].set_linewidth(0.5)
ax.spines["bottom"].set_visible(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.grid(False)

plt.tight_layout()
plt.savefig(
    f"{figdir}domains.pdf",
    format="pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# Save domain composition
ct.to_csv(f"{outdir}fig4b_domain_composition.csv", index=True)

## Compute domain dotplot

In [ ]:
t_cells = adata[adata.obs['celltype'] == 'T'].copy()

# Define domain groups
immune_rich_clusters = ["0", "4", "8", "9"]
neuroblast_rich_clusters = ["1", "2", "6", "7", "10", "12", "14", "16", "17", "18"]

def classify_3_domains(n):
    n_str = str(n)
    if n_str in immune_rich_clusters:
        return "Immune-rich"
    elif n_str in neuroblast_rich_clusters:
        return "Neuroblast-rich"
    else:
        return "Other"

# Apply classification
t_cells.obs["Spatial_Domain"] = t_cells.obs["neigh_kmeans"].apply(classify_3_domains)

# Define genes (same style as 5.find_domains)
genes_to_plot = [
    "CD4", "CD8A", "TRGC2",                    # lineage
    "CCR7", "SELL", "LEF1", "TCF7", "IL7R",           # Naive / Tcm / TLS
    "MKI67",                                   # Proliferation
    "FOS", "IFNG", "GZMH", "GZMB", "GZMK", "KLRD1",   # Cytotoxic
    "PDCD1", "LAG3", "TOX", "TIGIT",        # Exhaustion
]


In [ ]:
# Make dotplot
dp = sc.pl.dotplot(
    t_cells,
    genes_to_plot,
    groupby="Spatial_Domain",
    categories_order=["Immune-rich", "Neuroblast-rich", "Other"],
    cmap='YlGnBu',
    return_fig=True,
    show=False,
    standard_scale='var',
)

dp = dp.style(
    smallest_dot=0.01,
    largest_dot=70,
)

# Access axes and style borders
ax_dict = dp.get_axes()

# Make ticks thinner
for _, ax in ax_dict.items():
    if ax is None:
        continue
    ax.tick_params(axis="both", which="both", width=0.5, length=2)

# Main panel border
main_ax = ax_dict.get("mainplot_ax", None)
if main_ax is not None:
    for spine in main_ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.5)
        spine.set_color("#000000")

# Color legend bar border
cax = ax_dict.get("color_legend_ax", None)
if cax is not None:
    for spine in cax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.5)
        spine.set_color("#000000")


# Styling and save
fig = dp.fig
fig.set_size_inches(9/2.54, 3.5/2.54, forward=True)
fig.tight_layout()
fig.canvas.draw()

display(fig)

fig.savefig(f"{figdir}domains_dotplot_T.pdf", bbox_inches="tight")
plt.close(fig)

In [ ]:
domain_a = "Immune-rich"
domain_b = "Neuroblast-rich"

# Keep only genes explicitly listed in genes_to_plot
genes = [g for g in genes_to_plot if g in t_cells.var_names]
if len(genes) == 0:
    raise ValueError("No genes from genes_to_plot found in t_cells.var_names")

print(f"Using {len(genes)} genes from genes_to_plot.")

# Build expression dataframe (cells x genes)
X = t_cells[:, genes].X
if sparse.issparse(X):
    X = X.toarray()

expr_df = pd.DataFrame(X, columns=genes, index=t_cells.obs_names)
meta = t_cells.obs[["sample", "Spatial_Domain"]].copy()
dat = meta.join(expr_df)

# Keep only the two domains being tested
dat2 = dat[dat["Spatial_Domain"].isin([domain_a, domain_b])].copy()

# Per-sample pseudobulk mean expression per domain
pb = (
    dat2.groupby(["sample", "Spatial_Domain"])[genes]
        .mean()
        .reset_index()
)

# Stats per gene: paired Wilcoxon (sample-matched)
rows = []
for g in genes:
    wide = pb.pivot(index="sample", columns="Spatial_Domain", values=g)
    wide = wide.dropna(subset=[domain_a, domain_b])  # samples with both domains
    n_pairs = len(wide)

    if n_pairs < 3:
        rows.append({
            "gene": g,
            "n_pairs": n_pairs,
            "mean_Immune-rich": np.nan,
            "mean_Neuroblast-rich": np.nan,
            "median_delta_Neuro_minus_Immune": np.nan,
            "wilcoxon_W": np.nan,
            "p_value": np.nan
        })
        continue

    x = wide[domain_a].values
    y = wide[domain_b].values
    delta = y - x

    try:
        W, p = wilcoxon(y, x, alternative="two-sided", zero_method="wilcox", method="exact")
    except Exception:
        W, p = wilcoxon(y, x, alternative="two-sided", zero_method="wilcox")

    rows.append({
        "gene": g,
        "n_pairs": n_pairs,
        "mean_Immune-rich": float(np.mean(x)),
        "mean_Neuroblast-rich": float(np.mean(y)),
        "median_delta_Neuro_minus_Immune": float(np.median(delta)),
        "wilcoxon_W": float(W),
        "p_value": float(p)
    })

stats_df = pd.DataFrame(rows)

# BH-FDR correction across genes
valid = stats_df["p_value"].notna()
if valid.any():
    _, qvals, _, _ = multipletests(stats_df.loc[valid, "p_value"].values, method="fdr_bh")
    stats_df.loc[valid, "p_value_fdr_bh"] = qvals

# Optional star labels
def q_to_stars(q):
    if pd.isna(q):
        return ""
    if q < 1e-3:
        return "***"
    if q < 1e-2:
        return "**"
    if q < 5e-2:
        return "*"
    return ""

stats_df["stars"] = stats_df["p_value_fdr_bh"].apply(q_to_stars)

# Save
stats_df.to_csv(f"{outdir}fig4d_domain_expression_paired_wilcoxon_stats.csv", index=False)

display(stats_df.sort_values("p_value_fdr_bh", na_position="last"))

## Compute domain composition

In [ ]:
# Plot settings
domain_order = ["Immune-rich", "Neuroblast-rich", "Other"]
min_label_width = 0.04
figsize = (4/2.54, 7/2.54)
cmap = mpl.cm.get_cmap("YlGnBu", 5)

In [ ]:
# Plotting functions
def build_ct(df_obs):
    """Return proportion table: rows=Spatial_Domain, cols=celltypes_all."""
    df_t = df_obs[["Spatial_Domain", "celltypes_all"]].astype(str)

    ct = (
        df_t.groupby(["Spatial_Domain", "celltypes_all"])
            .size()
            .unstack(fill_value=0)
    )

    # proportions within each domain
    ct = ct.div(ct.sum(axis=1), axis=0).fillna(0)

    # enforce domain row order
    ct = ct.reindex(domain_order)

    # drop empty rows
    ct = ct.loc[ct.sum(axis=1) > 0]
    return ct

def plot_ct(ct, title, outpath, legend_title="celltypes_all"):
    if ct.empty:
        print(f"Skipping plot (empty table): {title}")
        return

    colors = [cmap(i % 5) for i in range(ct.shape[1])]

    fig, ax = plt.subplots(figsize=figsize)
    ct.plot.bar(
        stacked=True,
        color=colors,
        edgecolor="none",
        linewidth=0,
        width=0.8,
        ax=ax,
        legend=False
    )

    # labels on larger segments
    for p in ax.patches:
        h = p.get_height()
        if h > min_label_width:
            r, g, b, _ = p.get_facecolor()
            luminance = 0.299 * r + 0.587 * g + 0.114 * b
            text_color = "#000000" if luminance > 0.6 else "#FFFFD9"
            ax.text(
                p.get_x() + p.get_width() / 2,
                p.get_y() + h / 2,
                f"{h:.2f}",
                ha="center",
                va="center",
                fontsize=5,
                fontweight="bold",
                color=text_color
            )

    ax.set_title(title, fontsize=6)
    ax.set_xlabel("Domain", fontsize=6)
    ax.set_ylabel("Proportion", fontsize=6)

    # axis ticks styled like top10-neighbor plots
    x = np.arange(len(ct.index))
    ax.set_xticks(x)
    ax.set_xticklabels(ct.index, rotation=90, ha="right", fontsize=6)
    ax.set_xlim(-0.5, len(ct.index) - 0.5)

    ax.set_ylim(0, 1.0)
    yt = np.linspace(0, 1, 6)
    ax.set_yticks(yt)
    ax.set_yticklabels([f"{v:.1f}" for v in yt], fontsize=6)

    # clean style
    ax.set_frame_on(False)
    ax.grid(False)
    ax.spines["bottom"].set_linewidth(0.5)
    ax.spines["bottom"].set_visible(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="both", which="both", left=True, bottom=True, width=0.5, length=2)
    ax.patch.set_visible(False)
    fig.patch.set_visible(False)

    plt.tight_layout()
    plt.savefig(outpath, format="pdf", transparent=True, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
# Individual plots
samples = sorted([s for s in t_cells.obs["sample"].dropna().unique()])
print(f"Found {len(samples)} samples")

for sample in samples:
    t_sub = t_cells[t_cells.obs["sample"] == sample].copy()
    if t_sub.n_obs == 0:
        continue

    ct_sample = build_ct(t_sub.obs)
    safe_sample = str(sample).replace("/", "_").replace(" ", "_")
    plot_ct(
        ct_sample,
        title=f"T-cell celltypes_all by Spatial Domain ({sample})",
        outpath=f"{figdir}domains_T_subtypes_barplot_{safe_sample}.pdf"
    )

In [ ]:
# Consensus plot
ct_consensus = build_ct(t_cells.obs)
plot_ct(
    ct_consensus,
    title="T-cell celltypes_all by Spatial Domain (Consensus)",
    outpath=f"{figdir}domains_T_subtypes_barplot_consensus.pdf"
)

In [ ]:
# Make and save source data for 10 individual + 1 consensus domain plots
source_rows = []

# 10 individual plots
samples = sorted([s for s in t_cells.obs["sample"].dropna().unique()])
for sample in samples:
    t_sub = t_cells[t_cells.obs["sample"] == sample].copy()
    if t_sub.n_obs == 0:
        continue

    ct_sample = build_ct(t_sub.obs)
    if ct_sample.empty:
        continue

    long_df = (
        ct_sample.reset_index(names="Spatial_Domain")
        .melt(
            id_vars="Spatial_Domain",
            var_name="celltypes_all",
            value_name="Proportion"
        )
    )
    long_df["Plot_Type"] = "individual"
    long_df["Plot_ID"] = str(sample)
    source_rows.append(long_df)

# 1 consensus plot
ct_consensus = build_ct(t_cells.obs)
if not ct_consensus.empty:
    long_cons = (
        ct_consensus.reset_index(names="Spatial_Domain")
        .melt(
            id_vars="Spatial_Domain",
            var_name="celltypes_all",
            value_name="Proportion"
        )
    )
    long_cons["Plot_Type"] = "consensus"
    long_cons["Plot_ID"] = "consensus"
    source_rows.append(long_cons)

# Combine
source_data = pd.concat(source_rows, ignore_index=True)

source_data["Spatial_Domain"] = pd.Categorical(
    source_data["Spatial_Domain"], categories=domain_order, ordered=True
)

source_data = source_data[
    ["Plot_Type", "Plot_ID", "Spatial_Domain", "celltypes_all", "Proportion"]
].sort_values(["Plot_Type", "Plot_ID", "Spatial_Domain", "celltypes_all"])

source_data_wide = source_data.pivot_table(
    index=["Plot_Type", "Plot_ID", "Spatial_Domain"],
    columns="celltypes_all",
    values="Proportion",
    aggfunc="first",
    fill_value=0
).reset_index()

# Save
source_data_wide.to_csv(f"{outdir}fig5_domains_11plots_source_data_wide.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

# -------------------------------------------------
# 1) Build per-sample/domain/subtype proportions
# -------------------------------------------------
domain_order = ["Immune-rich", "Neuroblast-rich", "Other"]

df = t_cells.obs[["sample", "Spatial_Domain", "T_subtype"]].copy()
df = df[df["Spatial_Domain"].isin(domain_order)].copy()

# counts per sample x domain x subtype
ct = (
    df.groupby(["sample", "Spatial_Domain", "T_subtype"])
      .size()
      .reset_index(name="count")
)

# total T per sample x domain
tot = (
    df.groupby(["sample", "Spatial_Domain"])
      .size()
      .reset_index(name="total_t_in_domain")
)

prop = ct.merge(tot, on=["sample", "Spatial_Domain"], how="left")
prop["prop"] = prop["count"] / prop["total_t_in_domain"]

# Optional save source table
prop.to_csv(f"{outdir}fig4c_tsubtype_domain_prop_by_sample.csv", index=False)

# mean table (for text)
mean_tbl = (
    prop.groupby(["T_subtype", "Spatial_Domain"])["prop"]
        .mean()
        .unstack("Spatial_Domain")
        .reindex(columns=domain_order)
)
mean_tbl.to_csv(f"{outdir}fig4c_tsubtype_domain_mean_props.csv")
display(mean_tbl)

# -------------------------------------------------
# 2) Friedman omnibus per subtype (paired by sample)
# -------------------------------------------------
fried_rows = []
pair_rows = []

pairwise_domains = [
    ("Immune-rich", "Neuroblast-rich"),
    ("Immune-rich", "Other"),
    ("Neuroblast-rich", "Other"),
]

subtypes = sorted(prop["T_subtype"].dropna().unique())

for st in subtypes:
    dsub = prop[prop["T_subtype"] == st].copy()

    # wide table: rows=sample, cols=domain, values=proportion
    w = (
        dsub.pivot_table(index="sample", columns="Spatial_Domain", values="prop", aggfunc="mean")
            .reindex(columns=domain_order)
    )

    # Keep only samples with all 3 domains present (required for Friedman)
    w_complete = w.dropna(subset=domain_order).copy()
    n_complete = len(w_complete)

    if n_complete >= 3:
        stat_f, p_f = friedmanchisquare(
            w_complete["Immune-rich"].values,
            w_complete["Neuroblast-rich"].values,
            w_complete["Other"].values
        )
    else:
        stat_f, p_f = np.nan, np.nan

    fried_rows.append({
        "T_subtype": st,
        "n_complete_samples": int(n_complete),
        "friedman_chi2": float(stat_f) if pd.notna(stat_f) else np.nan,
        "friedman_p": float(p_f) if pd.notna(p_f) else np.nan,
        "mean_Immune-rich": float(w["Immune-rich"].mean()) if "Immune-rich" in w else np.nan,
        "mean_Neuroblast-rich": float(w["Neuroblast-rich"].mean()) if "Neuroblast-rich" in w else np.nan,
        "mean_Other": float(w["Other"].mean()) if "Other" in w else np.nan,
        "median_Immune-rich": float(w["Immune-rich"].median()) if "Immune-rich" in w else np.nan,
        "median_Neuroblast-rich": float(w["Neuroblast-rich"].median()) if "Neuroblast-rich" in w else np.nan,
        "median_Other": float(w["Other"].median()) if "Other" in w else np.nan,
    })

    # -------------------------------------------------
    # 3) Paired Wilcoxon post hoc (within-sample)
    # -------------------------------------------------
    for d1, d2 in pairwise_domains:
        wp = w[[d1, d2]].dropna().copy()
        n_pairs = len(wp)

        if n_pairs >= 3:
            # Two-sided paired Wilcoxon
            try:
                stat_w, p_w = wilcoxon(wp[d1].values, wp[d2].values,
                                       alternative="two-sided",
                                       zero_method="wilcox",
                                       method="exact")
            except Exception:
                stat_w, p_w = wilcoxon(wp[d1].values, wp[d2].values,
                                       alternative="two-sided",
                                       zero_method="wilcox")
        else:
            stat_w, p_w = np.nan, np.nan

        pair_rows.append({
            "T_subtype": st,
            "domain_1": d1,
            "domain_2": d2,
            "n_pairs": int(n_pairs),
            "wilcoxon_W": float(stat_w) if pd.notna(stat_w) else np.nan,
            "p_value": float(p_w) if pd.notna(p_w) else np.nan,
            "mean_1": float(wp[d1].mean()) if n_pairs else np.nan,
            "mean_2": float(wp[d2].mean()) if n_pairs else np.nan,
            "median_1": float(wp[d1].median()) if n_pairs else np.nan,
            "median_2": float(wp[d2].median()) if n_pairs else np.nan,
        })

fried_df = pd.DataFrame(fried_rows)
pair_df = pd.DataFrame(pair_rows)

# -------------------------------------------------
# 4) Multiple testing correction
# -------------------------------------------------
valid_f = fried_df["friedman_p"].notna()
if valid_f.any():
    _, qf, _, _ = multipletests(fried_df.loc[valid_f, "friedman_p"].values, method="fdr_bh")
    fried_df.loc[valid_f, "friedman_p_fdr_bh"] = qf

valid_p = pair_df["p_value"].notna()
if valid_p.any():
    _, qp, _, _ = multipletests(pair_df.loc[valid_p, "p_value"].values, method="fdr_bh")
    pair_df.loc[valid_p, "p_value_fdr_bh"] = qp

# -------------------------------------------------
# 5) Save outputs
# -------------------------------------------------
fried_df.to_csv(f"{outdir}fig4c_tsubtype_domain_friedman_stats.csv", index=False)
pair_df.to_csv(f"{outdir}fig4c_tsubtype_domain_paired_wilcoxon_stats.csv", index=False)

display(fried_df)
display(pair_df)

In [ ]:
# Save labeled adata
adata.write_h5ad(outdir + 'xenium_integrated_labeled.h5ad', compression='gzip')

## Extra code chunks for additional visualization
Not included in the manuscript

In [ ]:
# Visualizing domains in the context of spatial maps
samples = adata.obs["sample"].unique()

# Force natural numeric order for neigh_kmeans categories
nk = pd.to_numeric(adata.obs["neigh_kmeans"], errors="coerce")
order = sorted(nk.dropna().unique().astype(int))
ordered_cats = [str(i) for i in order]

adata.obs["neigh_kmeans"] = pd.Categorical(
    nk.astype("Int64").astype(str),
    categories=ordered_cats,
    ordered=True
)

# Assign RdBu (20) colors, one per neigh_kmeans category (in the same order)
cmap = mpl.cm.get_cmap("RdBu", 20)
colors = [mcolors.to_hex(cmap(i % 20)) for i in range(len(ordered_cats))]
adata.uns["neigh_kmeans_colors"] = colors

# Loop and Plot
for s in samples:
    fig, ax = plt.subplots(figsize=(6, 6))

    subset = adata[adata.obs["sample"] == s].copy()

    # enforce same category order + colors on subset
    subset.obs["neigh_kmeans"] = subset.obs["neigh_kmeans"].cat.set_categories(
        ordered_cats, ordered=True
    )
    subset.uns["neigh_kmeans_colors"] = adata.uns["neigh_kmeans_colors"]

    sc.pl.spatial(
        subset,
        color="neigh_kmeans",
        spot_size=30,
        title=f"Neighborhoods: {s}",
        ax=ax,
        show=False,
        legend_loc="right margin",
        frameon=False
    )

    plt.tight_layout()

    plt.savefig(f'{figdir}domains_{s}.png', 
        format='png', 
        transparent=True, 
        dpi=600,
        bbox_inches='tight')

    plt.show()

In [ ]:
# Make ribbon plot of celltype by domain
domain_order = ["9", "16", "2",]

# Build counts for celltype by domain
df = adata.obs[["neigh_kmeans", "celltype"]].astype(str)
ct = (
    df.groupby(["neigh_kmeans", "celltype"])
      .size()
      .unstack(fill_value=0)
)

# Keep only requested domains in order
ct = ct.reindex(domain_order).fillna(0)

# Convert to proportions
ct = ct.div(ct.sum(axis=1), axis=0)

# Sort ribbons by overall abundance across the selected domains
sorted_cols = ct.sum(axis=0).sort_values(ascending=False).index.tolist()
ct = ct[sorted_cols]

# X positions
x = np.arange(len(domain_order))

# colors (map per celltype)
color_map = {
    "Endothelial": "#d73027",
    "Fibroblast": "#f46d43",
    "Schwann": "#fdae61",
    "Neuroblast": "#fee090",
    "Macrophage": "#e0f3f8",
    "B": "#abd9e9",
    "T": "#74add1",
}

# Build color list aligned to ct.columns
colors = [color_map.get(c, "#CCCCCC") for c in ct.columns]

# Stacked area (ribbon) plot
fig, ax = plt.subplots(figsize=(4.5, 2.5))
ax.stackplot(
    x,
    ct.T.values,
    colors=colors,
    edgecolor="none"
)

# X labels in desired order
ax.set_xticks(x)
ax.set_xticklabels(domain_order)

# Legend on the right
ax.legend(
    ct.columns,
    title="celltype",
    bbox_to_anchor=(1.02, 0.5),
    loc="center left",
    frameon=False
)

# Style
ax.set_frame_on(False)
ax.grid(False)
for side in ["top", "right", "left", "bottom"]:
    ax.spines[side].set_visible(False)
ax.tick_params(left=False, bottom=False)

ax.set_ylabel("Proportion")
ax.set_title("Ribbon plot of celltype by domain")

plt.tight_layout()

plt.savefig(f'{figdir}domains_9162.png', 
    format='png', 
    transparent=True, 
    dpi=600,
    bbox_inches='tight')

plt.show()

In [ ]:
# Dotplot of T cell phenotype by spatial neighborhood
genes_to_plot = [
    'CD4', 'CD8A', 'TRGC2',  # general
    'CCR7', 'SELL', 'LEF1', 'TCF7',        # Naive / Tcm / TLS
    'IFNG', 'GZMH', 'GZMB', 'GZMK', 'KLRD1',  # Cytotoxic (fixed typo)
    'PDCD1', 'LAG3', 'HAVCR2', 'TOX',      # Exhaustion
    'MKI67'                                # Proliferation
]

valid_genes = [g for g in genes_to_plot if g in t_cells.var_names]

sc.pl.dotplot(
    t_cells,
    valid_genes,
    groupby='neigh_kmeans',
    standard_scale='var',
    cmap='Reds',
    title='T Cell Phenotype by Spatial Neighborhood',
)